# D1.2 · Context that makes triage work

**Function D — Security Operations → The SOC Analyst & Detection Engineer**  ·  *AI for Security*

Builds on **[D1.1 · From alert queue to loop operator](https://spbreed.github.io/cyber-commons/lessons/D1.1.html)**.

| | |
|---|---|
| Open-source tooling | Wazuh |
| Open-weight models | GLM-4.6, Llama 3.3 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


An alert about a human is triageable with three facts: who, what, when. An alert
about an agent needs three more, and without them every analyst has to guess.

- **The acting identity** and the principal it acted for (A2.1).
- **The scopes it held** at the time. This is the decisive field: reading
  `.env` is alarming for an agent scoped `repo:read` and routine for a
  secrets-rotation agent.
- **The delegation chain**, so the analyst can see who caused the task.

Without scope in the alert, the analyst's only options are to escalate
everything or to develop a habit of closing agent alerts. Both happen, and the
second one happens quietly.

## 2 · Demo — the same alert, with and without context

In [ ]:
import time
from dataclasses import dataclass, field

@dataclass
class Token:
    sub: str; actor: str; scopes: set; act: dict = None
    def chain(self):
        out, node = [], self.act
        while node: out.append(node["actor"]); node = node.get("act")
        c = list(reversed(out)) + [self.actor]
        if c[0] != self.sub: c.insert(0, self.sub)
        return c

AGENTS = {
 "patch-agent":   Token("dana@corp", "patch-agent", {"repo:read", "repo:write"},
                        {"actor": "orchestrator", "act": None}),
 "rotator-agent": Token("ops@corp", "rotator-agent", {"secrets:read", "secrets:write"},
                        {"actor": "scheduler", "act": None}),
}
EVENT = {"action": "read_file", "target": "/vault/.env", "ts": time.time()}

print("BARE ALERT (what most SOCs receive):")
for actor in AGENTS:
    print(f"   {actor} read {EVENT['target']}")
print("   → identical. An analyst cannot tell these apart.\n")

print("ENRICHED ALERT:")
for actor, tok in AGENTS.items():
    expected = "secrets:read" in tok.scopes
    print(f"   actor        {tok.actor}")
    print(f"   on behalf of {tok.sub}")
    print(f"   chain        {' → '.join(tok.chain())}")
    print(f"   scopes held  {sorted(tok.scopes)}")
    print(f"   verdict      {'EXPECTED — this agent rotates secrets' if expected else 'ANOMALY — no secrets scope'}")
    print()

## 3 · Where it breaks — measure the analyst's decision quality

In [ ]:
def triage_without_context(event):
    """All the analyst has is the action and the target."""
    return "escalate" if "/.env" in event["target"] or "secret" in event["target"] else "close"

def triage_with_context(event, token):
    needed = "secrets:read"
    if "/.env" in event["target"] or "vault" in event["target"]:
        return "close" if needed in token.scopes else "escalate"
    return "close"

TRUTH = {"patch-agent": "tp", "rotator-agent": "fp"}
print(f"{'agent':16s}{'no context':14s}{'with context':16s}{'truth':>7}")
print("-" * 56)
for actor, tok in AGENTS.items():
    a = triage_without_context(EVENT)
    b = triage_with_context(EVENT, tok)
    print(f"{actor:16s}{a:14s}{b:16s}{TRUTH[actor]:>7}")

print("\nWithout scopes, both escalate → the rotator generates a false positive")
print("every single night, and within a month the rule is tuned off.")

## 4 · The control — the six fields, and what each one decides

In [ ]:
FIELDS = {
 "acting identity":  "which agent — not the human whose token it borrowed",
 "principal":        "who the action was for",
 "delegation chain": "who caused the task; where to look for the trigger",
 "scopes held":      "THE decisive field — is this action within its remit?",
 "tool + target":    "what it did",
 "session/trace id": "so the analyst can pull the whole run (D1.5)",
}
for k, v in FIELDS.items(): print(f"{k:20s}{v}")

def enrich(event, token, trace_id):
    return {"acting_identity": token.actor, "principal": token.sub,
            "chain": " → ".join(token.chain()), "scopes": sorted(token.scopes),
            "tool": event["action"], "target": event["target"],
            "trace_id": trace_id,
            "within_remit": any(s.startswith("secrets") for s in token.scopes)
                            if "vault" in event["target"] or "/.env" in event["target"]
                            else True}

print()
for actor, tok in AGENTS.items():
    e = enrich(EVENT, tok, trace_id=f"tr-{actor[:4]}-8812")
    verdict = "close (within remit)" if e["within_remit"] else "ESCALATE (outside remit)"
    print(f"{actor:16s}{verdict}")
    print(f"{'':16s}{e['chain']}  scopes={e['scopes']}")

assert enrich(EVENT, AGENTS["patch-agent"], "x")["within_remit"] is False
assert enrich(EVENT, AGENTS["rotator-agent"], "x")["within_remit"] is True

## What you just proved

The bare alert is identical for both agents. Enriched, the secrets-rotation agent is within remit and the patch agent is not. Context-free triage escalates both — generating a nightly false positive — while scope-aware triage matches ground truth on both.

## Your turn

Check which of the six fields your agent telemetry carries today. Scopes-held is the one almost nobody logs, and it is the one that decides the alert.

---

**Next → [D1.3 · Agent-assisted detection engineering](https://spbreed.github.io/cyber-commons/lessons/D1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*